# 06 — Visualization & Qualitative Analysis

## What this notebook produces
1. Training vs validation loss curve
2. Source/target length histograms (already plotted in notebook 01)
3. Attention heatmap for a single example
4. 5 good qualitative examples
5. 5 bad qualitative examples with failure labels
6. Discussion: beam vs greedy, domain shift

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import csv
import numpy as np
import torch
import matplotlib.pyplot as plt

from configs.config import *
from src.visualization import plot_loss_curve, plot_attention_heatmap
from src.tokenizer.tokenizer_utils import UrduTokenizer
from src.data.dataset import QGenDataset
from src.model.encoder import Encoder
from src.model.decoder import Decoder
from src.model.seq2seq import Seq2Seq
from src.training.utils import load_checkpoint, get_device, set_seed
from src.decoding.greedy import greedy_decode

## 1. Loss Curve

In [ ]:
plot_loss_curve()
from IPython.display import Image
Image(filename=str(FIGURES_DIR / 'loss_curve.png'))

## 2. Attention Heatmap

We pick a readable validation example, generate its output via
greedy decoding, and visualise the attention weights.

In [ ]:
set_seed(SEED)
device = get_device()

tokenizer = UrduTokenizer(SP_MODEL_PATH)
vocab_size = tokenizer.vocab_size

enc_hidden = HIDDEN_SIZE * 2
encoder = Encoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, PAD_ID)
decoder = Decoder(vocab_size, EMBEDDING_DIM, HIDDEN_SIZE, enc_hidden, NUM_LAYERS, DROPOUT, PAD_ID)
model = Seq2Seq(encoder, decoder).to(device)
load_checkpoint(BEST_MODEL_PATH, model, device=device)

In [ ]:
valid_dataset = QGenDataset(VALID_FILE, tokenizer)

# Pick a short, readable example for the heatmap
example_idx = 0
item = valid_dataset[example_idx]
pair = valid_dataset.pairs[example_idx]

src_ids = torch.LongTensor([item['src_ids']]).to(device)
src_len = torch.LongTensor([item['src_length']])
src_mask = (src_ids == PAD_ID)

gen_ids, attn_weights = greedy_decode(model, src_ids, src_len, src_mask)
gen_text = tokenizer.decode([t for t in gen_ids if t != EOS_ID])

print(f'Source:    {pair["source"]}')
print(f'Reference: {pair["target"]}')
print(f'Generated: {gen_text}')

# Build attention matrix for heatmap
# attn_weights is a list of [S] tensors, one per generated token
src_pieces = tokenizer.encode(pair['source'])
# Add BOS/EOS labels
src_labels = ['<bos>'] + src_pieces + ['<eos>']
gen_pieces = [tokenizer.id_to_piece(t) for t in gen_ids if t != EOS_ID]

attn_matrix = torch.stack(attn_weights[:len(gen_pieces)]).numpy()
# Trim to match actual source length (excl padding)
actual_src_len = item['src_length']
attn_matrix = attn_matrix[:, :actual_src_len]
src_labels = src_labels[:actual_src_len]

plot_attention_heatmap(
    attn_matrix, src_labels, gen_pieces,
    save_path=FIGURES_DIR / 'attention_heatmap.png',
)

## 3. Qualitative Examples

We load the saved samples and identify 5 good and 5 bad examples.
Good/bad are determined by comparing generated text to reference.

**After real outputs are generated**, manually label bad examples with
failure categories: wrong question word, hallucinated entity, `<unk>`,
repetition, copied the sentence.

In [ ]:
# Load samples
samples = []
samples_path = RESULTS_DIR / 'samples.tsv'
if samples_path.exists():
    with open(samples_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            samples.append(row)
    print(f'Loaded {len(samples)} samples')
else:
    print('samples.tsv not found — run evaluation first')

In [ ]:
# Display first 10 samples for inspection
if samples:
    for i, s in enumerate(samples[:10]):
        print(f'--- Sample {i+1} ---')
        print(f'  Source:    {s["source"][:80]}...')
        print(f'  Reference: {s["reference"]}')
        print(f'  Greedy:    {s["greedy"]}')
        print(f'  Beam:      {s["beam"]}')
        print()

## 4. Qualitative analysis tables

After inspecting outputs, create `results/qualitative_examples.tsv`
with 5 good and 5 bad examples. Bad examples should be labelled
with failure categories.

**This section is populated after real model outputs exist.**

## 5. Discussion points (for the report)

- **Question-word performance**: Does the model learn common Urdu question
  words (کیا، کس، کتنا، کہاں) correctly?
- **Where beam helps**: Beam search may produce more fluent or complete
  questions by considering multiple hypotheses.
- **Where beam hurts**: Beam search can sometimes prefer generic, safe
  outputs over more specific but riskier greedy choices.
- **Wiki-UQA degradation**: Performance likely drops because the domain
  and vocabulary distribution differ from UQA training data.
- **Domain gap**: The UQA dataset contains translated data, which may
  introduce artifacts that the model learns but that don't generalise.